In [3]:
import cv2
import glob
import os
import numpy as np

image_folder = "data1"
output_folder = "results"
os.makedirs(output_folder, exist_ok=True)

image_files = glob.glob(f"{image_folder}/*.jpg")

for img_path in image_files:
    img = cv2.imread(img_path)
    if img is None:
        continue

    # Resize for consistency
    height, width = img.shape[:2]
    scale = 800 / max(height, width)
    img_resized = cv2.resize(img, (int(width*scale), int(height*scale)))

    gray = cv2.cvtColor(img_resized, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5,5), 0)
    
    # Adaptive threshold to highlight document
    thresh = cv2.adaptiveThreshold(blur, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                   cv2.THRESH_BINARY, 11, 2)
    
    # Edge detection
    edges = cv2.Canny(thresh, 50, 150)
    
    # Find contours
    contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    # Filter contours by area and aspect ratio
    best_cnt = None
    max_area = 0
    for cnt in contours:
        approx = cv2.approxPolyDP(cnt, 0.02 * cv2.arcLength(cnt, True), True)
        area = cv2.contourArea(approx)
        if len(approx) == 4 and area > max_area:
            x, y, w, h = cv2.boundingRect(approx)
            aspect_ratio = w / float(h)
            if 1.2 < aspect_ratio < 1.8:  # Typical Aadhaar aspect ratio ~ 1.5
                max_area = area
                best_cnt = approx

    if best_cnt is not None:
        cv2.drawContours(img_resized, [best_cnt], 0, (0, 255, 0), 3)

    # Save the result
    filename = os.path.basename(img_path)
    cv2.imwrite(os.path.join(output_folder, filename), img_resized)

print(f"Document detection completed. Check the folder: {output_folder}")


Document detection completed. Check the folder: results
